## 6.1 — Pricing Update (AWS only)

Downloads live AWS pricing for both platforms and upserts `price_per_unit` +
`units_per_hour` into `{catalog}.{schema}.results_prices` for **every supported
AWS region**. Region selection happens at dashboard/query time, not here.

### Scope

- **AWS only.** Azure and GCP are out of scope.
- **Snowflake:** `STANDARD_GEN_1` → `warehouse_size_type='gen1'`; `STANDARD_GEN_2` → `gen2`.
  `unit_label='credit'`. Tiers: Standard / Enterprise / Business Critical.
- **Databricks:** `SQL Serverless Compute` (what `0 - Job Setup` creates) →
  `warehouse_size_type='serverless'`. `unit_label='DBU'`. Tiers: Premium / Enterprise.

### Row multiplication

| Platform | Rows per run |
|---|---|
| Snowflake | `len(AWS_TO_SF_REGION) × (|sizes_gen1| + |sizes_gen2|)` — currently **19 × (10+8) = 342** |
| Databricks | `len(AWS_TO_DBX_REGION) × |sizes|` — currently **16 × 9 = 144** (minus regions where SQL Serverless isn't offered) |

### MERGE semantics

UPDATEs on match (this notebook is the authoritative source for prices) and INSERTs
new `(platform, warehouse_size, warehouse_size_type, region)` rows.

### Size mapping (DBX only — pricing JSON uses verbose names)

| JSON | results_prices |
|---|---|
| `2X-Small` | `2XS` |
| `X-Small`  | `XS`  |
| `Small`    | `S`   |
| `Medium`   | `M`   |
| `Large`    | `L`   |
| `X-Large`  | `XL`  |
| `2X-Large` | `2XL` |
| `3X-Large` | `3XL` |
| `4X-Large` | `4XL` |

Snowflake pricing JSON uses the short form already.


In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Constants ---
SF_PRICING_URL = (
    "https://www.snowflake.com/content/snowflake-site/global/en/pricing-options/"
    "calculator/_jcr_content/root/responsivegrid/container_397110202_/container/"
    "pricing_calculator.pricing.json"
)
DBX_PRICING_URL = "https://www.databricks.com/en-pricing-assets/data/pricing/AWS.json"

# AWS region ID → Snowflake region name in the pricing JSON (cloud=AWS branch)
AWS_TO_SF_REGION = {
    "us-east-1":      "US East (Northern Virginia)",
    "us-east-2":      "US East 2 (Ohio)",
    "us-west-2":      "US West (Oregon)",
    "ca-central-1":   "Canada Central",
    "eu-west-1":      "EU Dublin",
    "eu-west-2":      "Europe (London)",
    "eu-west-3":      "EU (Paris)",
    "eu-central-1":   "EU Frankfurt",
    "eu-north-1":     "Europe (Stockholm)",
    "sa-east-1":      "South America East 1 (São Paulo)",
    "ap-south-1":     "AP Mumbai",
    "ap-southeast-1": "AP Singapore",
    "ap-southeast-2": "AP Sydney",
    "ap-southeast-3": "Asia Pacific (Jakarta)",
    "ap-northeast-1": "AP Northeast 1 (Tokyo)",
    "ap-northeast-2": "Asia Pacific (Seoul)",
    "ap-northeast-3": "Asia Pacific (Osaka)",
    "af-south-1":     "Africa (Cape Town)",
    "me-central-1":   "Middle East (UAE)",
}

# AWS region ID → Databricks region name in AWS.json
AWS_TO_DBX_REGION = {
    "us-east-1":      "US East (N. Virginia)",
    "us-east-2":      "US East (Ohio)",
    "us-west-1":      "US West (N. California)",
    "us-west-2":      "US West (Oregon)",
    "ca-central-1":   "Canada",
    "eu-west-1":      "Europe (Ireland)",
    "eu-west-2":      "Europe (London)",
    "eu-west-3":      "Europe (France)",
    "eu-central-1":   "Europe (Frankfurt)",
    "eu-north-1":     "Europe (Stockholm)",
    "sa-east-1":      "SA (Brazil)",
    "ap-south-1":     "AP (Mumbai)",
    "ap-southeast-1": "AP (Singapore)",
    "ap-southeast-2": "AP (Sydney)",
    "ap-northeast-1": "AP (Tokyo)",
    "ap-northeast-2": "AP (Seoul)",
}

# Databricks pricing-JSON size label → benchmark short form
DBX_SIZE_MAP = {
    "2X-Small": "2XS",
    "X-Small":  "XS",
    "Small":    "S",
    "Medium":   "M",
    "Large":    "L",
    "X-Large":  "XL",
    "2X-Large": "2XL",
    "3X-Large": "3XL",
    "4X-Large": "4XL",
    # 5X-Large intentionally unmapped — benchmark doesn't cover it
}

DBX_COMPUTE = "SQL Serverless Compute"
SF_CLOUD    = "AWS"


In [ ]:
# --- Widgets ---
# (catalog + schema come from config.json via load_config() below)
# Tiers are NO LONGER selected here — this notebook now sweeps every supported
# tier per platform so the dashboard can pick which tier to compare at view time.
dbutils.widgets.removeAll()


In [ ]:
import json
from urllib.request import Request, urlopen
from decimal import Decimal

cfg = load_config()
catalog = cfg["dbx_catalog"]
schema  = cfg["dbx_schema"]

SF_TIERS  = ["Standard", "Enterprise", "Business Critical"]
DBX_TIERS = ["Premium", "Enterprise"]

if not catalog or not schema:
    raise RuntimeError(
        "Validation Error: config.json is missing databricks.catalog or databricks.schema."
    )

prices_table = f"{catalog}.{schema}.results_prices"

def fetch_json(url):
    req = Request(url, headers={"User-Agent": "Mozilla/5.0 (databricks-notebook)"})
    with urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())

print(f"Catalog/schema : {catalog}.{schema}  (from config.json)")
print(f"Snowflake tiers : {SF_TIERS}")
print(f"Databricks tiers: {DBX_TIERS}")
print(f"Target table   : {prices_table}")
print(f"Sweeping       : {len(AWS_TO_SF_REGION)} SF AWS regions, {len(AWS_TO_DBX_REGION)} DBX AWS regions")


## Snowflake pricing — all regions

In [ ]:
# --- Snowflake: per-region USD/credit (for the selected tier) + credits/hour by size ---
sf_pricing = fetch_json(SF_PRICING_URL)

def pick(entries, key, value):
    for e in entries:
        if e.get(key) == value:
            return e
    return None

credit_entry = pick(sf_pricing, "priceType", "Credit On Demand")
if not credit_entry:
    raise RuntimeError("Validation Error: 'Credit On Demand' block missing from Snowflake pricing JSON.")
cloud_entry = pick(credit_entry["data"], "cloud", SF_CLOUD)

# credits per hour — same across AWS regions (only varies by compute_type + size + cloud)
compute_entry = pick(sf_pricing, "priceType", "computeTypes")

def sf_credits_for(type_key):
    t = pick(compute_entry["data"], "type", type_key)
    if not t:
        return {}
    first = t["data"][0] if t["data"] else {}
    if "cloud" in first:
        cloud_data = pick(t["data"], "cloud", SF_CLOUD)
        rows = cloud_data.get("data", []) if cloud_data else []
    else:
        rows = t["data"]
    return {r["warehouse"]: float(r["creditsPerHour"]) for r in rows}

sf_gen1 = sf_credits_for("STANDARD_GEN_1")
# Gen 2 credits/hour are derived from Gen 1 by a fixed cloud-provider multiplier
# per the Snowflake Service Consumption Table (Table 1(b)). The JSON also exposes
# Gen 2 directly but we compute from Gen 1 here so the pricing source of truth
# stays the PDF (https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf).
# AWS / GCP: 1.35x;  Azure: 1.25x. We only support AWS regions today.
SF_GEN2_MULTIPLIER_AWS = 1.35
# Gen 2 only exists up to 4XL per Table 1(b); cap accordingly.
GEN2_SIZES = {"XS", "S", "M", "L", "XL", "2XL", "3XL", "4XL"}
sf_gen2 = {sz: round(cph * SF_GEN2_MULTIPLIER_AWS, 4)
           for sz, cph in sf_gen1.items() if sz in GEN2_SIZES}

# USD per credit per (region, tier)
# sf_region_prices[(aws_id, tier)] = price
sf_region_prices = {}
skipped_sf = []
for aws_id, sf_region_name in AWS_TO_SF_REGION.items():
    region_entry = pick(cloud_entry["data"], "region", sf_region_name)
    if not region_entry:
        skipped_sf.append((aws_id, "*", f"region '{sf_region_name}' not in SF pricing JSON"))
        continue
    for tier in SF_TIERS:
        tier_entry = pick(region_entry["data"], "productFamily2", tier)
        if not tier_entry:
            skipped_sf.append((aws_id, tier, f"tier '{tier}' not offered in {sf_region_name}"))
            continue
        sf_region_prices[(aws_id, tier)] = float(tier_entry["listPrice"])

print(f"Snowflake: {len(sf_region_prices)} (region, tier) entries priced, {len(skipped_sf)} skipped")
for (aws_id, tier), price in sorted(sf_region_prices.items()):
    print(f"  {aws_id:18s} {tier:18s} ${price}/credit")
for aws_id, tier, why in skipped_sf:
    print(f"  [skipped] {aws_id} / {tier}: {why}")
print(f"\ngen1 sizes ({len(sf_gen1)}): {sorted(sf_gen1.items(), key=lambda x: x[1])}")
print(f"gen2 sizes ({len(sf_gen2)}): {sorted(sf_gen2.items(), key=lambda x: x[1])}")


## Databricks pricing — all regions

In [ ]:
# --- Databricks: per-region baserate ($/DBU) for SQL Serverless + fixed dburate by size ---
dbx_pricing = fetch_json(DBX_PRICING_URL)
print(f"Databricks pricing JSON: {len(dbx_pricing)} entries")

# DBU-per-hour is the same across all AWS regions for SQL Serverless; only baserate varies.
# Capture (size → dburate) once from any region we find, and (region,size → baserate) per region.
# dbx_region_prices[(aws_id, tier)] = {size: {"baserate":..., "dburate":...}}
dbx_region_prices = {}
skipped_dbx = []

for aws_id, dbx_region_name in AWS_TO_DBX_REGION.items():
    for tier in DBX_TIERS:
        rows = [
            r for r in dbx_pricing
            if r.get("compute") == DBX_COMPUTE
            and r.get("region") == dbx_region_name
            and r.get("plan") == tier
            and r.get("instance") in DBX_SIZE_MAP
        ]
        if not rows:
            skipped_dbx.append((aws_id, tier, f"no '{DBX_COMPUTE}' rows for region '{dbx_region_name}' tier '{tier}'"))
            continue
        by_size = {}
        for r in rows:
            short = DBX_SIZE_MAP[r["instance"]]
            if short in by_size:
                continue
            by_size[short] = {
                "baserate": float(r["baserate"]),
                "dburate":  float(r["dburate"]),
            }
        dbx_region_prices[(aws_id, tier)] = by_size

print(f"\nDatabricks: {len(dbx_region_prices)} (region, tier) entries priced, {len(skipped_dbx)} skipped")
for (aws_id, tier), by_size in sorted(dbx_region_prices.items()):
    xs = by_size.get("XS") or by_size.get("2XS") or next(iter(by_size.values()))
    n_sizes = len(by_size)
    print(f"  {aws_id:18s} {tier:12s} {n_sizes} sizes; sample baserate=${xs['baserate']}/DBU")
for aws_id, tier, why in skipped_dbx:
    print(f"  [skipped] {aws_id} / {tier}: {why}")


## Build + MERGE

In [ ]:
# --- Build source DataFrame covering all regions ---
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType
from decimal import Decimal

rows = []

# Snowflake — gen1 + gen2 × every (region, tier)
for (aws_id, tier), usd_per_credit in sf_region_prices.items():
    for size, cph in sf_gen1.items():
        rows.append(("snowflake", size, "gen1", aws_id, tier,
                     Decimal(str(usd_per_credit)), Decimal(str(cph)), "credit"))
    for size, cph in sf_gen2.items():
        rows.append(("snowflake", size, "gen2", aws_id, tier,
                     Decimal(str(usd_per_credit)), Decimal(str(cph)), "credit"))

# Databricks — every (region, tier)
for (aws_id, tier), by_size in dbx_region_prices.items():
    for size, info in by_size.items():
        rows.append(("databricks", size, "serverless", aws_id, tier,
                     Decimal(str(info["baserate"])), Decimal(str(info["dburate"])), "DBU"))

src_schema = StructType([
    StructField("platform",            StringType(),       False),
    StructField("warehouse_size",      StringType(),       False),
    StructField("warehouse_size_type", StringType(),       False),
    StructField("region",              StringType(),       False),
    StructField("tier",                StringType(),       False),
    StructField("price_per_unit",      DecimalType(18, 6), False),
    StructField("units_per_hour",      DecimalType(18, 6), False),
    StructField("unit_label",          StringType(),       False),
])

src = spark.createDataFrame(rows, src_schema)
print(f"Built {src.count()} pricing rows across {src.select('region').distinct().count()} regions")
display(src.orderBy("platform", "region", "warehouse_size_type", "units_per_hour"))


In [ ]:
# --- MERGE INTO results_prices ---
src.createOrReplaceTempView("pricing_src")

merge_sql = f"""
MERGE INTO {prices_table} AS t
USING pricing_src AS s
ON t.platform            <=> s.platform
  AND t.warehouse_size      <=> s.warehouse_size
  AND t.warehouse_size_type <=> s.warehouse_size_type
  AND t.region              <=> s.region
  AND t.tier                <=> s.tier
WHEN MATCHED THEN UPDATE SET
  t.price_per_unit = s.price_per_unit,
  t.units_per_hour = s.units_per_hour,
  t.unit_label     = s.unit_label
WHEN NOT MATCHED THEN
  INSERT (platform, warehouse_size, warehouse_size_type, region, tier,
          price_per_unit, units_per_hour, unit_label)
  VALUES (s.platform, s.warehouse_size, s.warehouse_size_type, s.region, s.tier,
          s.price_per_unit, s.units_per_hour, s.unit_label)
"""

try:
    spark.sql(merge_sql)
    print(f"MERGE complete into {prices_table}")
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "does not exist" in str(e).lower():
        print(f"{prices_table} not found — creating with canonical schema")
        spark.sql(f"""
            CREATE TABLE {prices_table} (
                platform            STRING,
                warehouse_size      STRING,
                warehouse_size_type STRING,
                region              STRING,
                tier                STRING,
                price_per_unit      DECIMAL(18, 6),
                units_per_hour      DECIMAL(18, 6),
                unit_label          STRING
            ) USING DELTA
        """)
        spark.sql(merge_sql)
        print(f"MERGE complete into {prices_table}")
    else:
        raise


In [ ]:
# --- Verify: current prices, ordered by platform / region ---
display(
    spark.table(prices_table)
        .orderBy("platform", "region", "warehouse_size_type", "units_per_hour")
)


## Stage 3 — Build `tco_results` (duration + `test_price`)



Per-run duration aggregation (your `tco_results` definition) LEFT JOINed to `results_prices`

to add `test_price = duration_hours × units_per_hour × price_per_unit`.

Adaptive runs (`warehouse_type='gen2_adaptive'`) have no size price → `test_price` is NULL here

and is filled by `6.2 - Snowflake - Adaptive Pricing` from `warehouse_metering_history`.

In [ ]:
# --- Stage 3: build tco_results (per-run duration + priced test_price) ---
# Non-adaptive rows: test_price = duration_hours x units_per_hour x price_per_unit (results_prices).
# Adaptive rows (warehouse_type='gen2_adaptive') have no size price -> test_price NULL; filled by 6.2.
TCO_REGION = "us-east-1"
TCO_TIER   = "Enterprise"
queries_table = f"{catalog}.{schema}.results_queries"
tco_table     = f"{catalog}.{schema}.tco_results"

spark.sql(f"""
CREATE OR REPLACE TABLE {tco_table} AS
WITH r AS (
  SELECT
    platform, data_format, optimization_strategy,
    strategy_slug AS arch_ref, scale, warehouse_size,
    warehouse_size_type AS warehouse_type, test_type,
    ROUND(
      IF(test_type LIKE 'concurrency',
         (unix_timestamp(max(end_time)) - unix_timestamp(min(start_time))),
         sum(latency_ms) / 1000),
      0
    ) AS duration_seconds,
    benchmark_run_id, max_clusters
  FROM {queries_table}
  WHERE scale > 1 AND label_raw NOT LIKE 'drop_%'
  GROUP BY platform, scale, warehouse_size, test_type, data_format,
           optimization_strategy, benchmark_run_id, strategy_slug, warehouse_size_type, max_clusters
  HAVING count(*) IN (24, 103, 3518)
)
SELECT
  r.*,
  ROUND((r.duration_seconds / 3600.0) * (p.price_per_unit * p.units_per_hour), 4) AS test_price
FROM r
LEFT JOIN {prices_table} p
  ON  p.platform            = r.platform
  AND p.warehouse_size      = r.warehouse_size
  AND p.warehouse_size_type = r.warehouse_type
  AND p.region              = '{TCO_REGION}'
  AND p.tier                = '{TCO_TIER}'
""")
print(f"Built {tco_table}  (region={TCO_REGION}, tier={TCO_TIER})")
display(spark.sql(f"SELECT platform, arch_ref, warehouse_type, test_type, scale, duration_seconds, test_price FROM {tco_table} ORDER BY platform, arch_ref, test_type"))